# Frente A: señales (etapas 1 a 5)

Exploratorio, filtrado, artefactos ocular y cardíaco y segmentación. Las figuras se guardan en `informe/figuras/` con prefijo `a_`.

In [ ]:
# Entorno común a todos los notebooks: se corre desde codigo/ para que DIR_DATOS
# ("data" o $ESTADIOS_DATOS) apunte al mismo lugar que los scripts.
import os
import sys
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

# Si el paquete no está instalado con `pip install -e .`, se usa directamente codigo/src.
# Las dependencias (numpy, scipy, pandas, matplotlib, mne, scikit-learn) sí tienen que
# estar instaladas en el Python de este kernel.
try:
    import estadios  # noqa: F401
except ModuleNotFoundError:
    sys.path.insert(0, str(Path.cwd() / "src"))
FIGURAS = Path.cwd().parent / "informe" / "figuras"
FIGURAS.mkdir(parents=True, exist_ok=True)

import numpy as np
import matplotlib.pyplot as plt

from estadios.comun.config import Config
from estadios.comun.constantes import NOMBRES_ESTADIOS, N_ESTADIOS, SIN_ANOTAR
from estadios.datos.carga import ids_disponibles

USAR_DATOS_REALES = len(ids_disponibles()) > 0
print("Datos reales" if USAR_DATOS_REALES else "Sin datos descargados: se usan los sintéticos")

In [ ]:
def guardar(fig, nombre):
    """Guarda la figura en informe/figuras/ para incluirla con \\includegraphics."""
    fig.savefig(FIGURAS / f"{nombre}.pdf", bbox_inches="tight")

In [ ]:
from estadios.comun.constantes import ECG, EEG_F4
from estadios.comun.sintetico import registro_sintetico
from estadios.datos.carga import cargar_registro

registro = cargar_registro(ids_disponibles()[0]) if USAR_DATOS_REALES else registro_sintetico()
cfg = Config().preprocesamiento
print(registro.id, registro.n_epocas, "épocas")

## 1. Análisis exploratorio

In [ ]:
from estadios.preprocesamiento import exploratorio

f, por_estadio = exploratorio.espectro_por_estadio(registro, EEG_F4)
fig, ax = plt.subplots(figsize=(7, 4))
for estadio, psd in por_estadio.items():
    ax.semilogy(f, psd, label=NOMBRES_ESTADIOS[estadio])
ax.set(xlabel="Frecuencia (Hz)", ylabel="PSD (µV²/Hz)", xlim=(0, 40), title=EEG_F4)
ax.legend()
guardar(fig, "a_espectro_por_estadio")

In [ ]:
exploratorio.correlacion_eeg_eog(registro)

## 2. Filtrado FIR / IIR

In [ ]:
from estadios.preprocesamiento import filtrado
# TODO(Frente A): respuesta en frecuencia de ambos diseños y efecto sobre las ondas lentas.

## 3. Artefacto ocular (NLMS vs. Wiener)

In [ ]:
from estadios.preprocesamiento import ocular
# TODO(Frente A)

## 4. Latidos y artefacto cardíaco

In [ ]:
from estadios.preprocesamiento import cardiaco, qrs

latidos = qrs.detectar_latidos(registro.senales[ECG], registro.fs)
print(len(latidos), "latidos")

## 5. Segmentación

In [ ]:
from estadios.preprocesamiento import segmentacion
# TODO(Frente A). Ver la devolución: se puede reemplazar por ventanas fijas de 30 s.